In [1]:
import pandas as pd
import numpy as np
import warnings
from Bivariate import Bivariate
warnings.filterwarnings("ignore")

dataset = pd.read_csv("placement.csv")

dataset.drop('sl_no',inplace=True, axis=1)

quan, qual = Bivariate.quanQual(dataset)

### One way ANOVA Test

#### Comparing the Salary of students based on hsc_s [Commerce, Science, Arts]
#### 3 groups - [Commerce, Science, Arts]
#### 1 factor/independent variable - hsc_s

#### H0 -> Avg salaries of 3 groups are same
#### H1 -> Atleast one of the group salary is different

In [19]:
commerce_group_sal = dataset[dataset['hsc_s']=='Commerce']['salary']
arts_group_sal = dataset[dataset['hsc_s']=='Arts']['salary']
science_group_sal = dataset[dataset['hsc_s']=='Science']['salary']

print(commerce_group_sal.shape)
print(arts_group_sal.shape)
print(science_group_sal.shape)

(113,)
(11,)
(91,)


In [20]:
from scipy import stats

f_stat, p_val = stats.f_oneway(commerce_group_sal, arts_group_sal, science_group_sal)

print(f"F-statistic: {f_stat:.3f}")
print(f"P-value: {p_val:.3f}")
print(f"Commerce salary mean: {commerce_group_sal.mean():.3f}")
print(f"Arts salary mean: {arts_group_sal.mean():.3f}")
print(f"Science salary mean: {science_group_sal.mean():.3f}")
if p_val < 0.05:
    print("Reject H0: Average salary of 3 groups are significantly different")
else:
    print("Fail to reject H0: No significant difference in salary of 3 groups")

F-statistic: 0.969
P-value: 0.381
Commerce salary mean: 200938.053
Arts salary mean: 135636.364
Science salary mean: 203549.451
Fail to reject H0: No significant difference in salary of 3 groups


⚖️ Why p‑value > 0.05 Despite Arts Mean Being Lower

ANOVA compares variance, not just means.

    It looks at how much the group means differ relative to the variability within each group.

    If the Arts group has high variance (scores spread widely), then the difference in mean may not be statistically significant.

Sample size matters.

    With small sample sizes, even large mean differences may not be significant because the test has low statistical power.

    With large sample sizes, even small differences can become significant.

## TWO Way ANOVA test

#### Comparing salary of students based on hsc_s(Commerce, Science, Arts) and Gender(Male, Female)
#### 3 hsc_p * 2 gender = 6 groups [2 factors/independent variables]
#### 3 Hypothesis in Two way anova...
#### [Factor 1] => H0_hsc_s -> Avg Salary are same in all groups in hsc_s
#### [Factor 2] => H0_Gender -> Avg Salary are same in all groups in gender
#### [Interaction factor] => H0_Interaction -> No interaction b/w hsc_s and Gender, effect of hsc_s does not depend on Gender.

In [24]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

model = ols('salary ~ C(hsc_s) + C(gender) + C(hsc_s):C(gender)', data = dataset).fit()

anova_table = sm.stats.anova_lm(model, typ=2)

anova_table

,sum_sq,df,F,PR(>F)
C(hsc_s),3.446569e+10,2.0,0.731225,0.482546
C(gender),9.301217e+10,1.0,3.946696,0.048270
C(hsc_s):C(gender),6.183810e+10,2.0,1.311958,0.271501
Residual,4.925523e+12,209.0,NaN,NaN


TEST RESULT:

Avg salary of hsc_s are Same (Accept H0)

Avg salary of Gender are not same (Reject H0)

There is no interaction b/w hsc_s and gender (Accept H0)